<a href="https://colab.research.google.com/github/Woodpecker06/twinkletwinkle/blob/master/Gdriveconnect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import subprocess
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
import os
from tqdm.notebook import tqdm
from multiprocessing import Pool, cpu_count
from functools import partial

# Function to process a single video file
def process_video_file(video_filename, video_directory):
    video_path = os.path.join(video_directory, video_filename)
    audio_path = f"extracted_audio_{os.path.splitext(video_filename)[0]}.wav"
    spectrogram_image_path = os.path.join(video_directory, f"spectrogram_{os.path.splitext(video_filename)[0]}.png")

    try:
        # Skip if spectrogram already exists (this check is redundant after filtering but kept for robustness)
        if os.path.exists(spectrogram_image_path):
            return f"Skipped {video_filename}: Spectrogram already exists."

        # Extract audio via FFmpeg
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-vn", "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "1",
            audio_path
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Generate Spectrogram
        sample_rate, data = wavfile.read(audio_path)
        frequencies, times, Sxx = signal.spectrogram(data, fs=sample_rate, nperseg=1024)

        # Use the Agg backend for non-interactive plotting *within this function*
        matplotlib.use('Agg')
        fig = plt.figure(figsize=(12, 6))
        plt.pcolormesh(times, frequencies, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='inferno')
        plt.ylabel('Frequency (Hz)')
        plt.xlabel('Time (s)')
        plt.title(f'Spectrogram for {video_filename}')
        plt.colorbar(label='Intensity (dB)')
        plt.ylim(0, 10000)
        plt.savefig(spectrogram_image_path, dpi=300, bbox_inches='tight')
        plt.close(fig) # Close the specific figure to free up memory

        return f"Successfully processed {video_filename}"

    except subprocess.CalledProcessError as e:
        return f"Error processing {video_filename}: {e.stderr.decode().strip()}"
    except Exception as e:
        return f"An unexpected error occurred while processing {video_filename}: {e}"
    finally:
        # Ensure temporary audio file is removed even if other errors occur
        if os.path.exists(audio_path):
            os.remove(audio_path)


# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the directory where your video files are located
video_directory = "/content/drive/MyDrive/Cassandra Plot"

# 3. Get a list of all .mp4 files in the directory
all_mp4_files = [f for f in os.listdir(video_directory) if f.endswith('.mp4')]

if not all_mp4_files:
    print(f"No .mp4 files found in {video_directory}")
else:
    # Define a log file path for incremental progress
    log_file_path = os.path.join(video_directory, "processing_log.txt")
    print(f"Logging processing results to: {log_file_path}")

    processed_files_from_log = set()
    if os.path.exists(log_file_path):
        with open(log_file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith("Successfully processed "):
                    processed_filename = line[len("Successfully processed "):]
                    processed_files_from_log.add(processed_filename)
        print(f"Found {len(processed_files_from_log)} entries in the log file.")

    files_to_process = []
    skipped_count_log = 0
    skipped_count_spectrogram = 0

    for video_filename in all_mp4_files:
        if video_filename in processed_files_from_log:
            print(f"Skipping {video_filename}: Already marked as processed in log file.")
            skipped_count_log += 1
            continue

        spectrogram_image_path = os.path.join(video_directory, f"spectrogram_{os.path.splitext(video_filename)[0]}.png")
        if os.path.exists(spectrogram_image_path):
            print(f"Skipping {video_filename}: Spectrogram already exists.")
            skipped_count_spectrogram += 1
        else:
            files_to_process.append(video_filename)

    skipped_total = skipped_count_log + skipped_count_spectrogram

    if not files_to_process:
        print(f"All {len(all_mp4_files)} files in {video_directory} already have spectrograms or are marked in log. Nothing to process.")
    else:
        print(f"Found {len(all_mp4_files)} .mp4 files. {len(files_to_process)} files to process (skipped {skipped_total} existing). Processing in parallel...")
        num_processes = cpu_count() # Use all available CPU cores
        print(f"Using {num_processes} processes for parallel processing.")

        # Create a partial function with the fixed video_directory argument
        process_func = partial(process_video_file, video_directory=video_directory)

        with Pool(processes=num_processes) as pool:
            # Use imap_unordered for a dynamic progress bar and ordered results not required
            results = list(tqdm(pool.imap_unordered(process_func, files_to_process), total=len(files_to_process), desc="Processing Videos"))

        # Print all results and save them to a log file (append mode)
        with open(log_file_path, 'a') as log_file:
            for res in results:
                print(res)
                log_file.write(res + '\n')

print(f"All video files processed. Detailed log saved to {log_file_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logging processing results to: /content/drive/MyDrive/Cassandra Plot/processing_log.txt
Skipping Ring_FrontDoor_20260601_0100.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260601_0055.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260601_0049.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260601_0006.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260531_2323.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260531_2252.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260531_2248.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260531_1505.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260530_1624.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260529_1703.mp4: Spectrogram already exists.
Skipping Ring_FrontDoor_20260529_1120.mp4: Spectrogram already exists.
Sk

Processing Videos:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt
import os

# Get a list of all spectrogram image files
spectrogram_files = [f for f in os.listdir(video_directory) if f.startswith('spectrogram_') and f.endswith('.png')]

# Display the first spectrogram found, if any
if spectrogram_files:
    first_spectrogram_path = os.path.join(video_directory, spectrogram_files[0])
    img = plt.imread(first_spectrogram_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.title(f'Displaying: {spectrogram_files[0]}')
    plt.axis('off') # Hide axes for image display
    plt.show()
else:
    print(f"No spectrograms found in {video_directory}")

In [ ]:
import glob
import os

# Get a list of all temporary audio files generated
temp_audio_files = glob.glob('extracted_audio_*.wav')

# Remove each temporary audio file
for audio_file in temp_audio_files:
    try:
        os.remove(audio_file)
        print(f"Removed temporary audio file: {audio_file}")
    except OSError as e:
        print(f"Error removing {audio_file}: {e}")

print("Temporary audio files cleanup complete.")

In [ ]:
import glob

# Get a list of all temporary audio files generated
temp_audio_files = glob.glob('extracted_audio_*.wav')

# Remove each temporary audio file
for audio_file in temp_audio_files:
    try:
        os.remove(audio_file)
        print(f"Removed temporary audio file: {audio_file}")
    except OSError as e:
        print(f"Error removing {audio_file}: {e}")

print("Temporary audio files cleanup complete.")

In [ ]:
import subprocess

try:
    # Run ffmpeg -version and capture output
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True, check=True)
    print(result.stdout.splitlines()[0]) # Print only the first line (version info)
except FileNotFoundError:
    print("FFmpeg is not installed or not found in PATH.")
except subprocess.CalledProcessError as e:
    print(f"Error checking FFmpeg version: {e}")
    print(e.stderr)

In [ ]:
import os

# List the contents of the 'Cassandra Plot' directory in Google Drive
# This assumes the drive is mounted at /content/drive
!ls "/content/drive/MyDrive/Cassandra Plot"